# Research 39 — Fixed LightGBM variant blend screen

Compare the predeclared 80/20 current-reference blends for the DART and extra-trees candidates. Both component OOF matrices are aligned to `train.csv`; evaluate macro/weighted F1 and seed-42 fold slices. No fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), filename
    return probs / row_sums


p_base = load_probs("segment-oof.csv")
components = {
    "LightGBM DART VIP × region": "segment-lightgbm-dart-vip-region-oof.csv",
    "LightGBM extra-trees VIP × region": "segment-lightgbm-extra-trees-vip-region-oof.csv",
}
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
rows = []
blend_predictions = {}
for name, filename in components.items():
    component = load_probs(filename)
    for label, probs in [
        ("standalone", component),
        ("80/20 blend", 0.8 * p_base + 0.2 * component),
    ]:
        pred = probs.argmax(axis=1)
        row = {
            "candidate": name,
            "evaluation": label,
            "macro_f1": f1_score(y, pred, average="macro"),
            "weighted_f1": f1_score(y, pred, average="weighted"),
        }
        for fold, (_, valid_idx) in enumerate(folds, start=1):
            row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
        rows.append(row)
        if label == "80/20 blend":
            blend_predictions[name] = probs
results = pd.DataFrame(rows)
base_pred = p_base.argmax(axis=1)
base_row = {
    "candidate": "Current CatBoost/LightGBM reference",
    "evaluation": "reference",
    "macro_f1": f1_score(y, base_pred, average="macro"),
    "weighted_f1": f1_score(y, base_pred, average="weighted"),
}
for fold, (_, valid_idx) in enumerate(folds, start=1):
    base_row[f"fold_{fold}_macro_f1"] = f1_score(
        y[valid_idx], base_pred[valid_idx], average="macro"
    )
results = pd.concat([pd.DataFrame([base_row]), results], ignore_index=True)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-lightgbm-variant-screen.csv", index=False)
for name, probs in blend_predictions.items():
    stem = "dart" if "DART" in name else "extra-trees"
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / f"predictions/segment-lightgbm-{stem}-vip-region-blend-oof.csv", index=False
    )
print("Fixed 80/20 component-only blend comparison complete; Test not loaded.")

,candidate,evaluation,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1
0,Current CatBoost/LightGBM reference,reference,0.750895,0.764683,0.749445,0.749821,0.753390
1,LightGBM DART VIP × region,standalone,0.734562,0.746580,0.734291,0.733396,0.735984
2,LightGBM DART VIP × region,80/20 blend,0.750455,0.764007,0.748990,0.749387,0.752953
3,LightGBM extra-trees VIP × region,standalone,0.743154,0.754848,0.740474,0.743655,0.745307
4,LightGBM extra-trees VIP × region,80/20 blend,0.751649,0.765154,0.750324,0.751090,0.753508


Fixed 80/20 component-only blend comparison complete; Test not loaded.
